## 1. 라이브러리 로드

In [86]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import platform
import ast
from collections import Counter
import json

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

---
## 2. 데이터 로드

In [87]:
df_platinum_Match = pd.read_csv('./tft_game_dataset/TFT_Platinum_MatchData.csv')
df_diamond_Match = pd.read_csv('./tft_game_dataset/TFT_Diamond_MatchData.csv')
df_master_Match = pd.read_csv('./tft_game_dataset/TFT_Master_MatchData.csv')
df_grand_master_Match = pd.read_csv('./tft_game_dataset/TFT_GrandMaster_MatchData.csv')
df_challenger_Match = pd.read_csv('./tft_game_dataset/TFT_Challenger_MatchData.csv')

df_champion_info = pd.read_csv('./tft_game_dataset/TFT_Champion_CurrentVersion.csv')
df_items_info = pd.read_csv('./tft_game_dataset/TFT_Item_CurrentVersion.csv')


In [88]:
# 매치관련 데이터가 담긴 딕셔너리 정의
match_data = {
    'platinum': df_platinum_Match,
    'diamond': df_diamond_Match,
    'master': df_master_Match,
    'grand_master': df_grand_master_Match,
    'challenger': df_challenger_Match,
}

# 각 티어별 테이블에 'Tier'정보가 담긴 컬럼 추가
for name, df in match_data.items():
    df['tier'] = name


# 모든 티어의 경기데이터가 담긴 데이터프레임 제작
# ignore_index=True: 데이터를 병합한 뒤, 새로운 인덱스 부여
df_all_match = pd.concat(match_data.values(), ignore_index=True)

df_all_match.head(3)

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
0,KR_4291707834,1963.905273,6,27,5,1390.165771,"{'Cybernetic': 1, 'Demolitionist': 1, 'Infiltrator': 1, 'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Celestial': 1, 'Set3_Void': 1, 'Sniper': 1}","{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",platinum
1,KR_4291707834,1963.905273,8,37,3,1891.282715,"{'Blaster': 1, 'Chrono': 1, 'Cybernetic': 4, 'Demolitionist': 1, 'Rebel': 1, 'Set3_Blademaster': 2, 'Set3_Brawler': 1, 'Set3_Sorcerer': 1, 'Set3_Void': 1, 'Valkyrie': 1, 'Vanguard': 2}","{'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}",platinum
2,KR_4291707834,1963.905273,6,25,7,1279.461060,"{'Blaster': 1, 'Cybernetic': 1, 'DarkStar': 2, 'Infiltrator': 1, 'Mercenary': 1, 'Set3_Blademaster': 1, 'Set3_Mystic': 1, 'Valkyrie': 1}","{'Fiora': {'items': [1], 'star': 1}, 'Shaco': {'items': [6], 'star': 1}, 'Karma': {'items': [4], 'star': 1}, 'MissFortune': {'items': [3], 'star': 1}}",platinum


---
## 3. 데이터 전처리
### 3-1. 중복행 제거

In [89]:
# 중복행 제거
duplicates = df_all_match[df_all_match.duplicated(keep=False)]

print(f"중복 제거 전: {len(df_all_match)}")
print(f"중복된 행 개수: {len(duplicates)}")

# 결과 확인
df_all_match = df_all_match.drop_duplicates()
print(f"\n중복 제거 후: {len(df_all_match)}")

중복 제거 전: 399998
중복된 행 개수: 80

중복 제거 후: 399930


### 3-2. 컬럼명 소문자로 통일

In [90]:
# 전체 컬럼명 소문자 통일
df_all_match.columns = df_all_match.columns.str.lower()

print(df_all_match.columns)

Index(['gameid', 'gameduration', 'level', 'lastround', 'ranked',
       'ingameduration', 'combination', 'champion', 'tier'],
      dtype='str')


### 3-3. ranked=0인 데이터 삭제

In [91]:
# ranked = 0이 포함된 경기 데이터 삭제
df_all_match_2 = df_all_match.copy()

# ranked==0인 행의 gameId 추출
drop_game_ids = df_all_match_2[df_all_match_2['ranked']==0]['gameid'].unique()

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx)

print(f"ranked = 0인 데이터가 포함된 경기 데이터 삭제하기 전: {df_all_match.shape}")
print(f"ranked = 0인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

ranked = 0인 데이터가 포함된 경기 데이터 삭제하기 전: (399930, 9)
ranked = 0인 데이터가 포함된 경기 데이터 삭제한 후: (399886, 9)


### 3-4. 경기시간 = 0인 데이터 삭제

In [92]:
# 전체 게임시간 = 0인 행의 GameId 추출
zero_duration_ids = set(df_all_match_2[df_all_match_2['gameduration'] == 0]['gameid'])

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_2 = df_all_match_2[df_all_match_2['gameid'].isin(zero_duration_ids)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_2)

print(f"gameduration = 0인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

gameduration = 0인 데이터가 포함된 경기 데이터 삭제한 후: (399886, 9)


### 3-5. 경기 참여 유저 수가 8명 미만인 게임 삭제

In [93]:
# 게임 ID별 플레이어 수 정보 추가
df_all_match_2['player_cnt'] = df_all_match_2['gameid'].map(df_all_match_2['gameid'].value_counts())

# player_cnt < 8인 행의 gameId 추출
drop_game_ids_3 = df_all_match_2[df_all_match_2['player_cnt'] < 8]['gameid'].unique()

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_3 = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids_3)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_3)

print(f"player_cnt < 8인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

player_cnt < 8인 데이터가 포함된 경기 데이터 삭제한 후: (399872, 10)


### 3-6. 시즌2 데이터 삭제

In [94]:
# 시즌 2 고유 시너지 목록
season2_keys = {'Alchemist', 'Avatar', 'Berserker',
                'Crystal', 'Desert', 'Druid', 'Electric',
                'Light', 'Mage', 'Mountain', 'Mystic', 'Ocean',
                'Poison', 'Predator', 'Set2_Assassin', 'Set2_Blademaster',
                'Set2_Glacial', 'Set2_Ranger', 'Shadow', 'Soulbound',
                'Summoner', 'Warden', 'Wind', 'Woodland'}  # 시즌 2 시너지 입력


# 시즌 2 시너지가 하나라도 포함되면 시즌 2로 분류
df_all_match_2['season'] = df_all_match_2['combination'].apply(
    lambda x: 'season 2' if any(k in season2_keys for k in ast.literal_eval(x).keys())
    else 'season 3'
)

# season 2인 행의 gameId 추출
drop_game_ids_4 = df_all_match_2[df_all_match_2['season'] == 'season 2']['gameid'].unique()


# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_4 = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids_4)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_4)

print(f"season 2인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

season 2인 데이터가 포함된 경기 데이터 삭제한 후: (396640, 11)


### 3-7. 시즌 3 시너지 더미 데이터 ”TemplateTrait” 삭제
- 시너지 데이터 중 해당 값인 `{'TemplateTrait': 1}`만 삭제

In [95]:
# TemplateTrait 키가 있는 행만 필터링 후 값 확인
df_all_match_2[df_all_match_2['combination'].apply(
    lambda x: 'TemplateTrait' in ast.literal_eval(x).keys() # TemplateTrait 키가 있는 행만 필터링
)]['combination'].apply(                                    # 필터링된 행 중에서 combination 컬럼만 가져옴
    lambda x: ast.literal_eval(x).get('TemplateTrait')      # TemplateTrait 키에 할당된 값을 가져옴
).value_counts()


combination
1    65987
Name: count, dtype: int64

In [96]:
# 분석 목적에 영향을 주지 않는 dummy 데이터로 판단하여 시너지 중 'TemplateTrait'만 삭제
df_all_match_2['combination'] = df_all_match_2['combination'].apply(
    lambda x: json.dumps(
        # key: value = 키-값의 쌍을 의미함
        {key: value for key, value in ast.literal_eval(x).items() if key != 'TemplateTrait'},
        ensure_ascii=False
    )
)

In [97]:
# TemplateTrait 키가 있는 행 수 확인 → (0,11)이면 완전히 삭제된 것
df_all_match_2[df_all_match_2['combination'].apply(
    lambda x: 'TemplateTrait' in ast.literal_eval(x).keys()
)].shape

(0, 11)

### 3-8. 유저 ID 컬럼 제작

In [98]:
df_all_match_2['user_id'] = 'KR-USER-' + (df_all_match_2.index + 1).astype(str)

In [99]:
df_all_match_2['user_id']

0              KR-USER-1
1              KR-USER-2
2              KR-USER-3
3              KR-USER-4
4              KR-USER-5
               ...      
399993    KR-USER-399994
399994    KR-USER-399995
399995    KR-USER-399996
399996    KR-USER-399997
399997    KR-USER-399998
Name: user_id, Length: 396640, dtype: str

### 3-9. 티어가 섞인 경기 정보 추출

In [100]:
# 티어가 섞인 gameId 추출
mixed_gameids = df_all_match_2.groupby('gameid')['tier'].nunique() # gameid별로 tier의 고유값 개수를 계산
mixed_gameids = mixed_gameids[mixed_gameids > 1].index # 고유값이 2 이상인 gameId만 추출

# 해당 gameId의 티어 구성 확인
# mixed_gameids에 포함된 gameId를 가진 행만 추출 -> gameid별로 어떤 티어가 몇 명씩 있는지 카운트
df_all_match_2[df_all_match_2['gameid'].isin(mixed_gameids)].groupby('gameid')['tier'].value_counts()

gameid         tier        
KR_4263820118  platinum        8
               master          8
KR_4313697214  master          8
               platinum        8
KR_4320079059  platinum        8
               diamond         8
KR_4344513862  master          8
               diamond         8
KR_4347884427  diamond         8
               master          8
KR_4357966612  grand_master    8
               platinum        8
KR_4358922415  master          8
               diamond         8
KR_4361594426  master          8
               diamond         8
KR_4361773461  diamond         8
               grand_master    8
KR_4362846604  master          8
               diamond         8
KR_4364453473  grand_master    8
               diamond         8
KR_4365284161  master          8
               diamond         8
KR_4378896137  diamond         8
               platinum        8
KR_4381231118  diamond         8
               platinum        8
KR_4387025966  diamond         8
               

In [101]:
# 티어 순서 정의 (숫자가 높을수록 높은 티어)
tier_order = {
    'platinum': 1,
    'diamond': 2,
    'master': 3,
    'grand_master': 4,
    'challenger': 5
}

---
## 4. 티어믹스매치에서 상위 계급의 매치로 취급한 버전

### 4-1. 티어믹스매치에서 하위계급의 데이터 삭제

---
## 5. 티어믹스매치에서 하위 계급의 매치로 취급한 버전